# Stack Overflow Agentic RAG

An LLM searches Stack Overflow, selects relevant questions, reads their answers, and writes a response with source links.

**Before running:** use `uv sync --locked --extra notebook`, set `OLLAMA_API_KEY` in `.env`, and launch with `uv run --extra notebook jupyter lab notebooks/demo.ipynb`. Select the project’s `.venv` kernel.


In [4]:
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

# Locate .env when Jupyter starts at either the repository root or notebooks/.
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/rag_faq").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Open this notebook from inside the RAG-FAQ project.")
load_dotenv(project_root / ".env")

True

In [5]:
from rag_faq import DEFAULT_MODEL, AgenticRAG

api_key = os.getenv("LLM_API_KEY") or os.getenv("OLLAMA_API_KEY")
if not api_key:
    raise ValueError("Set OLLAMA_API_KEY or LLM_API_KEY in your project .env.")

llm_client = OpenAI(
    base_url=os.getenv("LLM_BASE_URL", "https://ollama.com/v1"),
    api_key=api_key,
    timeout=60,
    max_retries=0,
)
agent = AgenticRAG(
    llm_client,
    model=os.getenv("LLM_MODEL", DEFAULT_MODEL),
)

In [11]:
question = "How do I remove duplicates from a Python list while keeping their order?"  # noqa: E501
display(Markdown(agent.ask(question)))

Depending on your Python version and performance needs, there are several ways to remove duplicates from a list while preserving the original order.

### 1. For Python 3.7+ (The Recommended Way)
In Python 3.7 and later, regular dictionaries are guaranteed to maintain insertion order. The most concise and efficient way to remove duplicates is using `dict.fromkeys()`:

```python
items = [1, 2, 0, 1, 3, 2]
unique_items = list(dict.fromkeys(items))
# Result: [1, 2, 0, 3]
```
**Why it works:** `dict.fromkeys()` creates a dictionary where the list elements are the keys. Since keys must be unique, duplicates are removed; since dictionaries preserve order, the original sequence is maintained [2](https://stackoverflow.com/a/17016257).

### 2. For Python 3.6 and older
If you are using an older version of Python where `dict` does not guarantee order, you can use `collections.OrderedDict`:

```python
from collections import OrderedDict
items = [1, 2, 0, 1, 3, 2]
unique_items = list(OrderedDict.fromkeys(items))
# Result: [1, 2, 0, 3]
```
This functions identically to the `dict.fromkeys` method but ensures compatibility with older Python versions [2](https://stackoverflow.com/a/17016257).

### 3. High-Performance Approach (Custom Function)
If you are processing very large lists and want to maximize speed by avoiding some overhead, you can use a `set` to track seen elements within a list comprehension:

```python
def remove_duplicates(seq):
    seen = set()
    seen_add = seen.add
    return [x for x in seq if not (x in seen or seen_add(x))]

items = [1, 2, 0, 1, 3, 2]
unique_items = remove_duplicates(items)
# Result: [1, 2, 0, 3]
```
**How it works:** 
- It uses a `set` for $O(1)$ lookup time.
- The expression `(x in seen or seen_add(x))` works because `seen.add(x)` always returns `None`. Therefore, if `x` is not in `seen`, the `or` condition executes `seen_add(x)` and then evaluates to `False` (since `None` is falsy), allowing `not` to make it `True` so the item is kept in the list [1](https://stackoverflow.com/a/480227), [2](https://stackoverflow.com/a/17016257).
- Assigning `seen.add` to a local variable `seen_add` is a micro-optimization that reduces the cost of looking up the method on every iteration [1](https://stackoverflow.com/a/480227).

### Summary Comparison

| Method | Python Version | Complexity | Notes |
| :--- | :--- | :--- | :--- |
| `dict.fromkeys()` | 3.7+ | $O(n)$ | Simplest, fastest for most cases [2](https://stackoverflow.com/a/17016257). |
| `OrderedDict.fromkeys()` | 3.6- | $O(n)$ | Best for compatibility with older versions [2](https://stackoverflow.com/a/17016257). |
| Custom `set` loop | All | $O(n)$ | Potentially faster for massive datasets [1](https://stackoverflow.com/a/480227). |
| `more_itertools.unique_everseen` | All | $O(n)$ | Best if you need a **lazy iterator** or support for **non-hashable** items (like lists of lists) [2](https://stackoverflow.com/a/17016257). |

Sources:

1. [How do I remove duplicates from a list, while preserving order?](https://stackoverflow.com/a/480227) — Markus Jarderot, CC BY-SA 4.0
2. [How do I remove duplicates from a list, while preserving order?](https://stackoverflow.com/a/17016257) — jamylak, CC BY-SA 4.0

Edit `question` and rerun the last cell to ask something else. When finished, call `llm_client.close()` or shut down the kernel.
